In [1]:
# 02-vector-search hw

In [1]:
%%bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py

--2026-06-25 21:34:23--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

     0K .                                                     100%  164M=0s

2026-06-25 21:34:23 (164 MB/s) - ‘download.py’ saved [1376/1376]

--2026-06-25 21:34:24--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Le

In [3]:
! uv run python download.py

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


In [4]:
# Q1. Embedding a query

In [5]:
from embedder import Embedder
embed = Embedder()

q = "How does approximate nearest neighbor search work?"
v = embed.encode(q)
v[0] # Q1 -0.02

np.float64(-0.020582036807885073)

In [6]:
# Loading the data

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [8]:
documents[71] 

{'content': '# Chunking for Longer Texts\n\n<a href="https://www.youtube.com/watch?v=tyBRP_WewXA&list=PL3MmuxUbc_hIB4fSqLy_0AfTjVLpgjV3R">\n  <img src="https://markdown-videos-api.jorgenkh.no/youtube/tyBRP_WewXA">\n</a>\n\nOur FAQ data is well-structured: each document is a question-answer\npair. But what if your data is articles, transcripts, or slide\ndecks? You need to chunk it into pieces that are the right size\nfor embedding and retrieval.\n\n## Multiple articles\n\nIf you have multiple articles (blog posts, wiki pages, etc.\n\n):\n\n1. Assign each article a document ID\n2. Split each article into chunks\n3. Give each chunk a unique chunk ID (e.g., `doc_id_1`, `doc_id_2`)\n4. Evaluate retrieval with separate Hit Rate for both document ID\n   and chunk ID\n5. Tune chunk size using RAG evaluation metrics\n\n```json\n{\n  "doc_id": "abc123",\n  "chunk_id": "abc123_1",\n  "text": "first paragraph of the article..."\n}\n```\n\n## Single article or transcript\n\nIf you have one long pi

In [9]:
# Q2. Cosine similarity

In [10]:
for doc in documents[:1]:
    print(doc.keys())

dict_keys(['content', 'filename'])


In [11]:
target_doc = None

for doc in documents:
    if doc["filename"].endswith("07-sqlitesearch-vector.md"):
        target_doc = doc
        break

print(target_doc["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [12]:
content = target_doc["content"]
print(content[:50])

# Vector Search with sqlitesearch

Video: [Watch t


In [13]:
from embedder import Embedder
embed = Embedder()

q = "How does approximate nearest neighbor search work?"
v = embed.encode(q)
doc_vector = embed.encode(content)

In [14]:
len(doc_vector), doc_vector[0]

(384, np.float64(-0.006691674163480035))

In [15]:
v.dot(doc_vector)  #Q2 0.37

np.float64(0.361070280302606)

In [16]:
# Q3. Chunking and search by hand

In [17]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(documents), len(chunks)


(72, 295)

In [18]:
texts = [chunk["content"] for chunk in chunks]
for chunk in chunks:
    print(chunk["filename"])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/04-dataset.md
01-agentic-rag/lessons/04-dataset.md
01-agentic-rag/lessons/04-dataset.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/06-building-prompt.md
01-agentic-rag/lessons/06-building-prompt.md
01-agentic-rag/lessons/06-building-prompt.md
01-agentic-rag/lessons/07-llm.md
01-agentic-rag/lessons/07-llm.md
01-agentic-rag/lessons/07-llm.md
01-agentic

In [19]:
len(texts)

295

In [20]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [21]:
X, len(X)

(array([[-0.08756473,  0.01836385, -0.08122418, ...,  0.03053823,
         -0.02172771,  0.03277497],
        [ 0.02436193, -0.10619476,  0.03307318, ...,  0.01430081,
         -0.00125541,  0.04325696],
        [-0.01780485,  0.03103095,  0.00856106, ...,  0.0222022 ,
         -0.03375529,  0.04288229],
        ...,
        [ 0.00980343,  0.04912254,  0.01207489, ..., -0.09453995,
         -0.06321278,  0.04775798],
        [-0.03622024,  0.06821856, -0.01540897, ..., -0.00271628,
          0.01875559,  0.01007469],
        [-0.02975658, -0.00552574, -0.03531848, ...,  0.01044231,
          0.02297966, -0.01966068]], shape=(295, 384)),
 295)

In [22]:
q = "How does approximate nearest neighbor search work?"
v = embed.encode(q)

In [23]:
scores = X.dot(v)

In [24]:
len(scores)

295

In [25]:
idx = scores.argmax()

In [26]:
idx

np.int64(94)

In [27]:
chunks[idx]["filename"] # Q3 2-vector-search/lessons/07-sqlitesearch-vector.md

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [28]:
# Q4. Vector search with minsearch

In [29]:
from minsearch import VectorSearch
from sentence_transformers import SentenceTransformer
#model = SentenceTransformer("all-MiniLM-L6-v2")

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, chunks)

In [30]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [31]:
for result in results:
    print(result["filename"]) # Q4 04-evaluation/lessons/05-search-metrics.md

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md


In [32]:
# Q5. Text search vs vector search

In [33]:
from minsearch import Index

In [52]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)
query = "How do I store vectors in PostgreSQL?"
text_results = index.search(
    query,
    num_results=5
)
for r in text_results:
    print(r["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [63]:
query = "How do I store vectors in PostgreSQL?"

vector_query =  embed.encode(query)
vector_results = vindex.search(vector_query, num_results=5)

for r in vector_results:
    print(r["filename"])  #Q5 02-vector-search/lessons/08-pgvector.md


02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [64]:
# Q6. Hybrid search  Reciprocal Rank Fusion (RRF)
# RRF(d) = sum over lists of  1 / (k + rank(d))
# "Sum over lists" means we go through every ranked list and, 
# for each list where the document appears, add its 1 / (k + rank) contribution.
# A document found by both searches collects a score from each list,
# while one found by only a single search collects just one.

In [65]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [69]:
query = "How do I give the model access to tools?"

text_results = index.search(
    query,
    num_results=5
)

In [71]:
text_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [72]:
query_vector = embed.encode(query)

vector_results = vindex.search(
    query_vector,
    num_results=5
)

In [73]:
vector_results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [74]:
results = rrf([vector_results, text_results])

In [75]:
for result in results:
    print(result["filename"])  # Q6 01-agentic-rag/lessons/13-function-calling.md

01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
